# Driver Drowsiness Detection Project

## Introduction
Driver drowsiness is a significant factor contributing to road accidents worldwide. This project aims to develop a deep learning-based system for detecting driver drowsiness in real time, which can help reduce accidents caused by fatigue. By utilizing Convolutional Neural Networks (CNNs), the system classifies driver states as either **Drowsy** or **Non-Drowsy** based on facial images.

## Dataset Used: Driver Drowsiness Dataset (DDD)
The **Driver Drowsiness Dataset (DDD)** is a collection of cropped face images of drivers, extracted from the **Real-Life Drowsiness Dataset** videos. The dataset was processed using VLC software to extract frames, and the Viola-Jones algorithm was used to detect and crop the region of interest (faces). The dataset was utilized for training and testing CNN architectures for driver drowsiness detection.

### Dataset Properties
- **Image Type:** RGB  
- **Number of Classes:** 2 (**Drowsy** and **Non-Drowsy**)  
- **Image Size:** 227 × 227 pixels  
- **Total Images:** 41,790+  
- **File Size:** 2.32 GB  


In [ ]:
# %pip install numpy
# %pip install pandas
# %pip install tensorflow
# %pip install shutil
# %pip install scikit-learn
# %pip install distutils
# %pip install Pillow
# %pip install seaborn

%pip install pandas
%pip install openpyxl

In [ ]:
import os
import shutil
import tensorflow as tf
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import pandas as pd

from sklearn.metrics import confusion_matrix, classification_report, roc_auc_score, matthews_corrcoef, f1_score
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Dense, Dropout, GlobalAveragePooling2D
from tensorflow.keras.models import Model, load_model
from tensorflow.keras.regularizers import l2
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.model_selection import train_test_split

## Configuration & Directories Setup

In [ ]:
DATA_DIR = os.path.join(os.getcwd(), "Driver Drowsiness Dataset (DDD)")
DROWSY_DIR = os.path.join(DATA_DIR, 'Drowsy')
NON_DROWSY_DIR = os.path.join(DATA_DIR, 'Non Drowsy')
OUTPUT_DIR = "prepared_data"

# Validate dataset availability
assert os.path.exists(DROWSY_DIR), "Drowsy folder not found"
assert os.path.exists(NON_DROWSY_DIR), "Non-drowsy folder not found"

# Create dataset partitions
for split in ["train", "val", "test"]:
    for category in ["drowsy", "non_drowsy"]:
        os.makedirs(os.path.join(OUTPUT_DIR, split, category), exist_ok=True)

## Data Preprocessing & Splitting

In [ ]:
def split_and_copy(source_dir, target_dirs, split_ratios=(0.8, 0.1, 0.1)):
    filenames = os.listdir(source_dir)
    train, temp = train_test_split(filenames, test_size=split_ratios[1] + split_ratios[2], random_state=42)
    val, test = train_test_split(
        temp,
        test_size=split_ratios[2] / (split_ratios[1] + split_ratios[2]),
        random_state=42
    )

    category_name = os.path.basename(source_dir).lower().replace(" ", "_")
    for split_name, files in zip(["train", "val", "test"], [train, val, test]):
        category_path = os.path.join(target_dirs[split_name], category_name)
        os.makedirs(category_path, exist_ok=True)
        for file in files:
            shutil.copy(os.path.join(source_dir, file), os.path.join(category_path, file))

# Run once to prepare the dataset
split_and_copy(DROWSY_DIR, {
    "train": os.path.join(OUTPUT_DIR, "train"),
    "val": os.path.join(OUTPUT_DIR, "val"),
    "test": os.path.join(OUTPUT_DIR, "test")
})

split_and_copy(NON_DROWSY_DIR, {
    "train": os.path.join(OUTPUT_DIR, "train"),
    "val": os.path.join(OUTPUT_DIR, "val"),
    "test": os.path.join(OUTPUT_DIR, "test")
})

In [ ]:
train_datagen = ImageDataGenerator(
    rescale=1.0/255.0,
    rotation_range=30,
    zoom_range=0.3,
    width_shift_range=0.3,
    height_shift_range=0.3,
    shear_range=0.3,
    horizontal_flip=True,
    fill_mode="nearest"
)

val_datagen = ImageDataGenerator(rescale=1.0/255.0)

train_generator = train_datagen.flow_from_directory(
    os.path.join(OUTPUT_DIR, "train"),
    target_size=(96, 96),
    batch_size=32,
    class_mode='binary'
)

val_generator = val_datagen.flow_from_directory(
    os.path.join(OUTPUT_DIR, "val"),
    target_size=(96, 96),
    batch_size=32,
    class_mode='binary'
)

## Model Architecture

In [ ]:
base_model = MobileNetV2(weights='imagenet', include_top=False, input_shape=(96, 96, 3), alpha=0.35)
base_model.trainable = False

x = GlobalAveragePooling2D()(base_model.output)
x = Dropout(0.5)(x)
x = Dense(64, activation='relu', kernel_regularizer=l2(0.01))(x)
x = Dropout(0.4)(x)
output = Dense(1, activation='sigmoid')(x)

model = Model(inputs=base_model.input, outputs=output)

## Evaluation Function

In [ ]:
eval_results = []

def evaluate_model(model, test_generator, epoch):
    y_true = test_generator.classes
    y_pred_prob = model.predict(test_generator)
    y_pred = (y_pred_prob > 0.5).astype(int).flatten()

    # Confusion matrix
    cm = confusion_matrix(y_true, y_pred)
    tn, fp, fn, tp = cm.ravel()

    accuracy = (tp + tn) / (tp + tn + fp + fn)
    sensitivity = tp / (tp + fn) if (tp + fn) != 0 else 0  # Recall
    specificity = tn / (tn + fp) if (tn + fp) != 0 else 0
    precision = tp / (tp + fp) if (tp + fp) != 0 else 0
    fnr = fn / (fn + tp) if (fn + tp) != 0 else 0
    fpr = fp / (fp + tn) if (fp + tn) != 0 else 0
    auc = roc_auc_score(y_true, y_pred_prob)
    mcc = matthews_corrcoef(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)

    eval_results.append({
        "Epoch": epoch,
        "TP": tp, "TN": tn, "FP": fp, "FN": fn,
        "Accuracy": accuracy,
        "Sensitivity": sensitivity,
        "Specificity": specificity,
        "Precision": precision,
        "False Negative Rate": fnr,
        "False Positive Rate": fpr,
        "AUC": auc,
        "Matthews CorrCoef": mcc,
        "F1-Score": f1
    })

    plt.figure(figsize=(6, 4))
    sns.heatmap(
        cm,
        annot=True,
        fmt='d',
        cmap='Blues',
        xticklabels=['Non-Drowsy', 'Drowsy'],
        yticklabels=['Non-Drowsy', 'Drowsy']
    )
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.title(f'Confusion Matrix - Epoch {epoch}')
    plt.show()

    print("Classification Report:")
    print(classification_report(y_true, y_pred, target_names=['Non-Drowsy', 'Drowsy']))

## TFLite Conversion

In [ ]:
def convert_and_save_tflite_model(model, save_path="drowsiness_detection_model_quantized.tflite"):
    converter = tf.lite.TFLiteConverter.from_keras_model(model)
    converter.optimizations = [tf.lite.Optimize.DEFAULT]
    tflite_quantized_model = converter.convert()

    with open(save_path, "wb") as f:
        f.write(tflite_quantized_model)

    print(f"TFLite model saved to {save_path}")

## Model Compilation & Training

In [ ]:
epochs = [10,20,30,40,50,100]

def train_save_results():
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )

    test_datagen = ImageDataGenerator(rescale=1.0/255.0)
    test_generator = test_datagen.flow_from_directory(
        os.path.join(OUTPUT_DIR, "test"),
        target_size=(96, 96),
        batch_size=32,
        class_mode='binary',
        shuffle=False
    )

    lr_scheduler = ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=3,
        min_lr=1e-7,
        verbose=1
    )

    for i, e in enumerate(epochs):
        model.fit(
            train_generator,
            validation_data=val_generator,
            epochs=e,
            steps_per_epoch=train_generator.samples // train_generator.batch_size,
            validation_steps=val_generator.samples // val_generator.batch_size,
            callbacks=[lr_scheduler]
        )

        test_loss, test_acc = model.evaluate(test_generator)
        print(f"Test Accuracy after {e} epochs: {test_acc * 100:.2f}%")

        evaluate_model(model, test_generator, epoch=e)
        model.save(f"drowsiness_detection_model_{e}.h5")

        if i == len(epochs) - 1:
            convert_and_save_tflite_model(model)

    df_results = pd.DataFrame(eval_results)
    df_results.to_excel("evaluation_metrics.xlsx", index=False)
    print("Evaluation metrics saved to evaluation_metrics.xlsx")

In [ ]:
train_save_results()

## Final Model Size

In [ ]:
final_model_path = "drowsiness_detection_model_100.h5"
if os.path.exists(final_model_path):
    size_in_MB = os.path.getsize(final_model_path) / (1024 * 1024)
    print(f"Model Size: {size_in_MB:.2f} MB")
else:
    print("Final model file not found yet. Run training first.")